# Swarm Demo — Multi-Agent Log Analysis with Memory Pointer Pattern

Based on: [Solving Context Window Overflow in AI Agents](https://arxiv.org/html/2511.22729v1) — IBM Research, 2024

## The Problem

In a multi-agent workflow, agents need to share large datasets. Passing 145KB of logs as a message between agents overflows each agent's context window. The data needs to exist somewhere all agents can access — without any of them holding it in their context.

## The Solution: `invocation_state`

`tool_context.invocation_state` is the Strands API for sharing data across agents in a swarm. Tools write large data there and return only a pointer key. Any agent in the swarm can read that key from `invocation_state` — the raw data never enters any LLM context window.

```
agent.state         → scoped to one agent (single-agent pattern)
invocation_state    → shared across all agents in the swarm (multi-agent pattern)
```

## The Tools

Five tools in `tools.py` built on `ToolContext.invocation_state` (the agents are created in this notebook):

| Tool | What it does | Writes to invocation_state | Reads from invocation_state |
|------|-------------|---------------------------|------------------------------|
| `fetch_logs_swarm(app_name, hours)` | Generates log events, stores them | `"logs-{app_name}"` | — |
| `analyze_errors_swarm(logs_pointer)` | Counts errors by service | `"error_analysis"` | `logs_pointer` |
| `detect_latency_swarm(logs_pointer)` | Calculates p95 latency | `"latency_analysis"` | `logs_pointer` |
| `generate_report_swarm()` | Combines both analyses | — | `"error_analysis"`, `"latency_analysis"` |
| `get_error_details_swarm(logs_pointer, service)` | Drills into errors for one service | — | `logs_pointer` |

## Architecture

```
┌───────────────────────────────────────────────────────────────────┐
│                        invocation_state                            │
│  "logs-payment-service"  →  [600 events, 145KB]                   │
│  "error_analysis"        →  {total_errors, by_service, ...}       │
│  "latency_analysis"      →  {p95_latency_ms, anomalies_count, ...}│
└───────┬──────────────────────┬────────────────────────────────────┘
   write│               read+write│                         read│
        ▼                         ▼                              ▼
┌──────────────┐   ┌──────────────────────┐   ┌──────────────────────┐
│  Collector   │──►│       Analyzer       │──►│      Reporter        │
│ fetch_logs() │   │ analyze_errors()     │   │ generate_report()    │
│              │   │ detect_anomalies()   │   │                      │
└──────────────┘   └──────────────────────┘   └──────────────────────┘
```

## What We Test

| Step | What it shows |
|------|---------------|
| 1 — Run the Swarm | Collector → Analyzer → Reporter pipeline, data flowing via invocation_state |
| 2 — Follow-up investigation | Investigator agent reuses stored data after swarm completes — no re-fetch |

## Setup: install dependencies

Run the cell below once to install everything from `requirements.txt`. This demo needs **strands-agents 1.56.0+**. Restart the kernel afterward if prompted.

In [1]:
%pip install -r requirements.txt

import importlib.metadata as _m
_v = _m.version("strands-agents")
assert tuple(int(x) for x in _v.split(".")[:2]) >= (1, 56), (
    f"strands-agents {_v} is too old. This demo needs >= 1.56.0. "
    "Re-run the install cell and restart the kernel."
)
print(f"strands-agents {_v} — OK")


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
strands-agents 1.56.0 — OK


In [2]:
import os
import logging
import warnings

# Silence OpenTelemetry noise. Strands instruments tool calls with OTel; in threaded
# multi-agent runs (Swarm) the context-detach can log 'Failed to detach context'
# tracebacks. They are harmless, but disabling the SDK and raising the logger level
# keeps the notebook output clean.
os.environ['OTEL_SDK_DISABLED'] = 'true'
logging.getLogger('opentelemetry').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore', message='Failed to detach context')

from dotenv import load_dotenv
load_dotenv()

from strands import Agent
from strands.multiagent import Swarm
# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands.models.openai import OpenAIModel

# Swarm tools built on ToolContext.invocation_state (defined in tools.py)
from tools import (
    fetch_logs_swarm,
    analyze_errors_swarm,
    detect_latency_swarm,
    generate_report_swarm,
    get_error_details_swarm,
)

if not os.getenv('OPENAI_API_KEY'):
    raise ValueError(
        'OPENAI_API_KEY not set. Get your API key from https://platform.openai.com/api-keys '
        'then add OPENAI_API_KEY=your-key to a .env file.'
    )

MODEL = OpenAIModel(model_id="gpt-4o-mini")

# ─────────────────────────────────────────────────────────────────────────────
# How to switch the model provider (token counting works the same on all of them).
#
# Amazon Bedrock — uses boto3, NO extra package needed.
#   Requires AWS credentials and model access enabled in the Amazon Bedrock console.
#       from strands.models import BedrockModel
#       MODEL = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0", region_name="us-east-1")
#   Docs: https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/
#
# Anthropic (direct API) — requires:  pip install 'strands-agents[anthropic]'
#   The API key goes inside client_args (get one at https://console.anthropic.com/).
#       from strands.models.anthropic import AnthropicModel
#       MODEL = AnthropicModel(client_args={"api_key": os.getenv("ANTHROPIC_API_KEY")},
#                              model_id="claude-sonnet-4-6", max_tokens=1028)
#   Docs: https://strandsagents.com/docs/user-guide/concepts/model-providers/anthropic/
# ─────────────────────────────────────────────────────────────────────────────

# ── Create the three specialized agents (each only gets its own tools) ──────────
collector = Agent(
    name="collector",
    description="Fetches application logs and stores them in shared state for analysis",
    system_prompt=(
        "You collect data. Fetch logs with fetch_logs_swarm, then hand off to analyzer. "
        "Do NOT hand off to reporter."
    ),
    tools=[fetch_logs_swarm],
    model=MODEL,
)

analyzer = Agent(
    name="analyzer",
    description="Analyzes error patterns and latency anomalies from logs in shared state",
    system_prompt=(
        "You analyze data. Call analyze_errors_swarm AND detect_latency_swarm "
        "with the pointer from the collector (e.g. 'logs-payment-service'). "
        "You MUST call both tools before handing off. After both complete, hand off to reporter."
    ),
    tools=[analyze_errors_swarm, detect_latency_swarm],
    model=MODEL,
)

reporter = Agent(
    name="reporter",
    description="Generates the final incident report from analyses in shared state",
    system_prompt=(
        "You write reports. Call generate_report_swarm to produce the final report. "
        "After generating the report, present it to the user. Do NOT hand off to other agents."
    ),
    tools=[generate_report_swarm],
    model=MODEL,
)

# ── Build the Swarm (collector → analyzer → reporter) ───────────────────────────
swarm = Swarm(
    [collector, analyzer, reporter],
    entry_point=collector,
    max_handoffs=6,
    max_iterations=10,
)

print("✅ Swarm ready: collector → analyzer → reporter")


/Users/eliaws/.pyenv/versions/3.11.7/lib/python3.11/site-packages/pydantic/plugin/_schema_validator.py:39: UserWarning: ImportError while loading the `logfire-plugin` Pydantic plugin, this plugin will not be installed.

ImportError("cannot import name 'ReadableLogRecord' from 'opentelemetry.sdk._logs' (/Users/eliaws/.pyenv/versions/3.11.7/lib/python3.11/site-packages/opentelemetry/sdk/_logs/__init__.py)")
  plugins = get_plugins()


✅ Swarm ready: collector → analyzer → reporter


---
## Step 1 — Run the Swarm

One request triggers the full Collector → Analyzer → Reporter pipeline. While it runs, notice:

- Each agent calls only its own tools — the Collector never calls `generate_incident_report`, the Reporter never calls `fetch_logs`
- The LLM in each agent sees pointer strings like `"logs-payment-service"`, not 145KB of raw JSON
- `node_history` in the result shows the handoff order

> Note: 6 hours × 100 events/hour = 600 log events. This will take ~20–30 seconds.

In [3]:
print("⏳ Running swarm (~20–30s)...\n")

# Explicit shared state dict. We pass it as invocation_state so the swarm's tools
# write the logs/analyses into it — and it stays available for the follow-up in Step 2.
shared_state = {}

result = swarm(
    "Fetch 6 hours of logs for payment-service, analyze errors and latency, then generate an incident report.",
    invocation_state=shared_state,
)

print(f"\nStatus:     {result.status}")
print(f"Agents:     {' → '.join(n.node_id for n in result.node_history)}")
print(f"Time:       {result.execution_time}ms")

# 📊 Token counting in a Swarm: each agent has its OWN metrics.
# We iterate over result.results (collector → analyzer → reporter) and sum
# each one's accumulated_usage to see the TOTAL cost of the multi-agent system.
swarm_tokens = 0
for _nid, _nres in result.results.items():
    if _nres.result.metrics:
        swarm_tokens += _nres.result.metrics.accumulated_usage['totalTokens']
print(f"💰 Tokens (all agents): {swarm_tokens:,} total")

# Data the swarm stored in shared_state — reused (not re-fetched) in Step 2
_data_keys = [k for k in ('logs-payment-service', 'error_analysis', 'latency_analysis') if k in shared_state]
print(f"📦 shared_state holds: {_data_keys}")


⏳ Running swarm (~20–30s)...




Tool #1: fetch_logs_swarm



Tool #2: handoff_to_agent


I have successfully fetched 6

 hours of logs for the payment-service and handed them off to the analyzer for error and latency analysis. The task is now in the capable hands of the

...

 generated. Here are the details:

### Incident Report Summary
- **Total Errors**: 156
- **Error Rate**

: 26.0%
- **P95 Latency**: 4767 ms
- **Anomalies Count**: 29

### Errors by Service

 
- **Cache Layer**: 39 errors
- **Auth Service**: 42 errors
- **API Gateway**: 40 errors
- **DB

 Connector**: 35 errors

### Recommendations
- **HIGH**: Error rate exceeds 5%
- **MEDIUM**: High latency anomaly count

If

 you need further assistance or additional analysis, feel free to ask!
Status:     Status.COMPLETED
Agents:     collector → analyzer → reporter
Time:       9753ms
💰 Tokens (all agents): 4,325 total
📦 shared_state holds: ['logs-payment-service', 'error_analysis', 'latency_analysis']


---
## Step 2 — Follow-up Investigation

The swarm completed — but the `shared_state` dict we passed in still holds everything: the 145KB of logs, the error analysis, and the latency analysis. An investigator agent can now ask follow-up questions **without re-fetching any data**.

This is the key advantage over a single-agent approach: in a traditional system, each follow-up question would require re-fetching 145KB of logs. By reusing the same `shared_state` dict (passed as `invocation_state`), the data persists across calls and any agent can access it at any time.

In [4]:
from strands import Agent

# A new agent for follow-up questions. It reuses the SAME shared_state from Step 1,
# so the 145KB of logs and the analyses are already there — nothing is re-fetched.
investigator = Agent(
    name="investigator",
    system_prompt=(
        "You investigate incidents. The logs pointer is 'logs-payment-service'. "
        "Use get_error_details_swarm to drill into specific services."
    ),
    tools=[get_error_details_swarm, analyze_errors_swarm],
    model=MODEL,
)

inv_tokens = 0
print("👤 Turn 1: Which service had the most errors?\n")
_r = investigator(
    "Based on the error analysis in shared state, which service had the most errors? "
    "The logs are at 'logs-payment-service'",
    invocation_state=shared_state,
)
# 📊 Token counting — Strands native metric (same for OpenAI, Bedrock, etc.).
# result.metrics.accumulated_usage sums ALL the LLM calls made during this task:
#   inputTokens  = tokens sent (prompt + system prompt + tool definitions + history)
#   outputTokens = tokens generated by the model
#   totalTokens  = input + output
# We check '.metrics' because it can be None if the provider does not report usage.
inv_tokens += _r.metrics.accumulated_usage['totalTokens'] if _r.metrics else 0

print("\n👤 Turn 2: Show me the actual error logs for that service\n")
_r = investigator(
    "Show me 3 detailed error logs for the service with the most errors",
    invocation_state=shared_state,
)
inv_tokens += _r.metrics.accumulated_usage['totalTokens'] if _r.metrics else 0

print("\n👤 Turn 3: What status codes are those errors?\n")
_r = investigator(
    "What HTTP status codes are those errors returning?",
    invocation_state=shared_state,
)
inv_tokens += _r.metrics.accumulated_usage['totalTokens'] if _r.metrics else 0

print("\n📦 Data persisted in shared_state throughout — never re-fetched")
print(f"💰 Tokens (3 turns, data reused): {inv_tokens:,} total")


👤 Turn 1: Which service had the most errors?




Tool #1: analyze_errors_swarm


The service with the most errors based on the analysis is the **auth-service**, which had a total of **42 errors**.


👤 Turn 2: Show me the actual error logs for that service




Tool #2: get_error_details_swarm


...

)
     at module11.function11(file11.py:35)
     at module12.function12(file12.py:48)
     at module13.function13(file13.py

:73)
     at module14.function14(file14.py:88)
     ```
👤 Turn 3: What status codes are those errors?



The HTTP status codes for the errors in the **auth-service** are as follows:

1. **Error Log 1:** Status Code **503**


2. **Error Log 2:** Status Code **404**
3. **Error Log 3:** Status Code **500** 

So, the returned status codes are **503**, **404**,

 and **500**.
📦 Data persisted in shared_state throughout — never re-fetched
💰 Tokens (3 turns, data reused): 8,582 total


---
## Key Takeaways

1. **Swarm handles coordination** — collector → analyzer → reporter with autonomous handoffs
2. **`invocation_state` for multi-agent data** — the official Strands API for sharing data across agents in a swarm
3. **Large data stays out of context** — 145KB+ of logs in `invocation_state`, only pointers in LLM context
4. **Data persists after swarm completes** — follow-up investigation reuses the same stored data without re-fetching
5. **Same ToolContext API** — single-agent uses `agent.state`, multi-agent uses `invocation_state`, both via `ToolContext`

## References

- [Strands Swarm](https://strandsagents.com/docs/user-guide/concepts/multi-agent/swarm/) — Multi-agent orchestration
- [Shared State Across Multi-Agent Patterns](https://strandsagents.com/docs/user-guide/concepts/multi-agent/multi-agent-patterns/) — invocation_state for data sharing
- [Strands ToolContext](https://strandsagents.com/docs/user-guide/concepts/tools/custom-tools/) — Accessing agent.state and invocation_state
- [Solving Context Window Overflow](https://arxiv.org/html/2511.22729v1) — IBM Research
- [Towards Effective GenAI Multi-Agent Collaboration](https://arxiv.org/pdf/2412.05449) — Amazon, payload referencing
- [Code Repository](https://github.com/aws-samples/sample-why-agents-fail)